In [1]:
import pandas as pd
import numpy as np
from udrud_framework import calculate_k_max, detect_plateau_and_estimate, calculate_ud_rud_metrics

# Load your empirical dataset
df = pd.read_csv("2021_2025_disposable_income.csv")

# equivalize income and weight
df["weight"] = df["weight"] * df["size"]
df["income"] = df["income"] / np.sqrt(df["size"])

# Setup parameters
k_min = 20
p = 0.05  # tail depth (5% of sample size)
window_size = 5

# Initialize empty Python lists to store the row dictionaries
summary_list = []
estimates_list = []
indices_list = []

for year, group in df.groupby("year"):
    summary_list.append({
        "year": year,
        "No Households": group["income"].count(),
        "No Negative Incomes": np.sum(group["income"] < 0),
        "Minimum Income": np.min(group["income"])
    })

    y_val = group["income"]
    w_val = group["weight"]

    # 1. Calculate K_max using the helper function
    k_max = calculate_k_max(y_val, w_val, p)

    # Ensure k_range is valid
    k_max = max(k_min + 1, k_max)
    k_range_obj = range(k_min, k_max)

    # 2. Execute Level 3 Optimizer
    gamma, best_k = detect_plateau_and_estimate(group["income"], group["weight"], k_range_obj, window_size)

    # 3. Store threshold estimates
    estimates_list.append({"year": year, "k^*": best_k, "gamma^": gamma})

    # 4. Calculate final metrics
    metrics = calculate_ud_rud_metrics(group["income"], group["weight"], gamma)

    # Simply add the year to the dictionary calculated in step 4
    metrics["year"] = year
    indices_list.append(metrics)

# Convert lists to DataFrames for viewing the results.
estimate_df = pd.DataFrame(estimates_list)
summary_df = pd.DataFrame(summary_list)
indices_df = pd.DataFrame(indices_list)

print(summary_df)
print()

print(estimate_df)
print()

print(indices_df)


   year  No Households  No Negative Incomes  Minimum Income
0  2021          18187                   68   -21976.000000
1  2022          17954                   81    -5584.022251
2  2023          18094                   84   -16155.975737
3  2024          18314                   87   -12746.306838
4  2025          18664                   66    -5998.000000

   year   k^*        gamma^
0  2021   988 -38180.703019
1  2022  1032  -5853.680013
2  2023  1039 -21090.098056
3  2024  1086 -18489.749215
4  2025   219  -6102.060326

         Gamma^         mu_y    Gini_y      CV_y       mu_z       V_z  \
0 -38180.703019  3352.658847  0.335438  0.791447  12.388186  0.626388   
1  -5853.680013  3561.954416  0.334957  0.700568   2.643390  0.490795   
2 -21090.098056  3762.478842  0.330319  0.701516   6.605373  0.492125   
3 -18489.749215  4093.699809  0.329079  0.698464   5.516635  0.487853   
4  -6102.060326  4277.484235  0.330719  0.703539   2.426554  0.494967   

     Gini_z      CV_z       L1_